# 0.1 - Data Collection-1

- SpaceX REST API - launch data, rocket used, payload delivered, launch specs, landing specs, landing outcome.

In [98]:
import pandas as pd
import numpy as np
import requests
import json

# Pandas setting to completely show table
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(url)
response.raise_for_status()

HTTPError: 525 Server Error: <none> for url: https://api.spacexdata.com/v4/launches/past

It's a Cloudflare error...

The server of SpaceX REST API is down, so I had to look for other sources. But the static JSON provided by the course for project use still proved to be useful for retrieving corse data -- `gridfins_True`, `gridfins_False`, `legs_True`, `legs_False`. Furthermore, with the static JSON, I was still able to follow the time period of launches that is considered in the project.

## Data Collection from other sources

![falcon-9-components](../references/falcon-9-scale-and-transportation-infographic-v0-AJTKrBYPWvz-es3BQYfmOhiqpFl72jFH-IUBGRT43LM.webp)

The following is the list of features that are needed:
- **Booster Version** - `str`; The rocket engine used for the launch, e.g., Falcon Heavy, Falcon 9.
- **Payload Mass** - `float`; The total mass (in kg) of cargo carried by the spacecraft during the launch, e.g., 800 kg
- **Orbit** - `str`; The type of orbit in which the spacecraft is placed upon launch, e.g., Sun-synchronous orbit (SSO), Low Earth orbit (LEO).
- **LaunchSite** - `str`; name of the launch site, e.g., VAFB SLC 4E
- **Outcome** - `str`; Concatenation of landing success (`bool`) and landing type ('str'); e.g., 'True ASDS'. ASDS means autonomous spaceport drone ship. 
- **Flights** - `int`; number of flights with the core used in a launch
- **GridFins** - `bool`; whether or not the spacecraft has grid fins.
- **Reused** - `bool`; whether or not the spacecraft is reused for other flights.
- **Legs** - `bool`; whether or not the spacecraft has landing legs.
- **LandingPad** - `str`; name of the landing pad, e.g., LZ-2
- **Block** - `int`; version of the rocket, e.g., Falcon 9 block 5
- **ReusedCount** - `int`; number of times the rocket is reused.
- **Serial** - `str`; identification of the booster used, e.g., B1051
- **Longitude** - `float`; longitude of the launchpad.
- **Latitude** - `float`; latitude of the launchpad.

Features needed, the reference where data is retrieved, and the corresponding item in the reference.

| Feature | Reference (API Endpoint / JSON / TSV) | Item |
| --- | --- | --- |
| `BoosterVersion` | [LL2 API previous launches](https://ll.thespacedevs.com/docs/#/launches/launches_previous_list) | `rocket.configuration.name` |
| `Orbit` | [LL2 API previous launches](https://ll.thespacedevs.com/docs/#/launches/launches_previous_list) | `mission.orbit.name`|
| `Payload mass` | [General Catalog of Artificial Space Objects](https://planet4589.org/space/gcat/web/launch/lcols.html) | `OrbPay`|
| `LaunchSite` | [LL2 API previous launches](https://ll.thespacedevs.com/docs/#/launches/launches_previous_list) | `pad.name` |
| `Outcome` | [LL2 API previous launches](https://ll.thespacedevs.com/docs/#/launches/launches_previous_list) | `rocket.launcher_stage.landing.success` & `rocket.launcher_stage.landing.type.abbrev`|
| `Flights` | [LL2 API previous launches](https://ll.thespacedevs.com/docs/#/launches/launches_previous_list) | `rocket.launcher_stage.launcher.flights` |
| `Gridfins` | [Static JSON provided by the course for project use](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json) | `cores.gridfins` |
| `Reused` | [LL2 API previous launches](https://ll.thespacedevs.com/docs/#/launches/launches_previous_list) | `rocket.launcher_stage.launcher.reused` | 
| `Legs` | [Static JSON provided by the course for project use](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json) | `cores.legs` |
| `LandingPad` | [LL2 API previous launches](https://ll.thespacedevs.com/docs/#/launches/launches_previous_list) | `rocket.launcher_stage.landing.location.abbrev` |
| `Block` | [LL2 API previous launches](https://ll.thespacedevs.com/docs/#/launches/launches_previous_list) | `rocket.configuration.variant` |
| `ReusedCount` | [LL2 API previous launches](https://ll.thespacedevs.com/docs/#/launches/launches_previous_list) | `(rocket.launcher_stage.launcher_flight_number) - 1` |
| `Serial` | [LL2 API previous launches](https://ll.thespacedevs.com/docs/#/launches/launches_previous_list) | `rocket.launcher_stage.launcher.serial_number` |
| `Longitude` | [LL2 API previous launches](https://ll.thespacedevs.com/docs/#/launches/launches_previous_list) | `pad.longitude` |
| `Latitude` | [LL2 API previous launches](https://ll.thespacedevs.com/docs/#/launches/launches_previous_list) | `pad.latitude` |

### Data from Launch Library 2 (LL2) API

Since these are API calls, a script was made to retrieve them and to cache each call as JSON files saved in `data/raw`

In [81]:
from pathlib import Path
import time

import requests


def get_ll2_launches(offset):
    ll2_api = "https://ll.thespacedevs.com/2.3.0/launches/previous/"
    query_params = dict(
        mode="detailed", limit=100, rocket__configuration__name="Falcon 9", offset=offset
    )

    for attempt in range(3):
        try:
            response = requests.get(ll2_api, params=query_params)
            if response.status_code == 504:
                print("504 Server Error. Retrying in {} seconds...".format(2 * (attempt + 1)))
                time.sleep(2 * (attempt + 1))
                continue
            response.raise_for_status()
            return response.content

        except requests.exceptions.HTTPError as e:
            print("Error: {}".format(e))
            return None

    print("Failed after 3 attempts (504 Server Error)")
    return None

def download_all_ll2_launches():
    for offset in range(0, 700, 100):
        parent_dir = Path.cwd().parent
        file_dir = parent_dir / "data" / "raw"
        file_dir.mkdir(parents=True, exist_ok=True)
        file_name = "ll2-api-2.3.0-launches-previous-{}.json".format(offset)
        file_path = file_dir / file_name
        if file_path.is_file():
            print("{} already exists.".format(file_name))
            continue
        try:
            data = get_ll2_launches(offset)
            if not isinstance(data, bytes):
                raise TypeError("Expected bytes, got {}".format(type(data)))
            with open(file_path, "wb") as f:
                f.write(data)
        except Exception as e:
            print("Error: {}".format(e))
            break

In [83]:
download_all_ll2_launches()

ll2-api-2.3.0-launches-previous-0.json already exists.
ll2-api-2.3.0-launches-previous-100.json already exists.
ll2-api-2.3.0-launches-previous-200.json already exists.
ll2-api-2.3.0-launches-previous-300.json already exists.
ll2-api-2.3.0-launches-previous-400.json already exists.
ll2-api-2.3.0-launches-previous-500.json already exists.
ll2-api-2.3.0-launches-previous-600.json already exists.


### Data provided by course

The launch data provided by the course and as well as data from Jonathan McDowell's GCAT are considered static files and were downloaded with the following utility function I wrote:

In [99]:
def download_launch_data_static(url, file_name):
    parent_dir = Path.cwd().parent
    file_dir = parent_dir / "data" / "raw"
    file_dir.mkdir(parents=True, exist_ok=True)
    file_path = file_dir / file_name

    if file_path.is_file():
        print("{} already exists".format(file_name))
        return None
    
    response = requests.get(url)
    
    try:
        response.raise_for_status()
        
    except requests.exceptions.HTTPError as e:
        print("Error: {}".format(e))
        return None

    try:
        with open(file_path, "wb") as f:
            f.write(response.content)
    except Exception as e:
        print("Error: {}".format(e))
        return None

In [104]:
url_ibm='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'
file_name_ibm = 'ibm-ds-capstone-launch-data.json'
download_launch_data_static(url_ibm, file_name_ibm)

ibm-ds-capstone-launch-data.json already exists


### Data from Jonathan McDowell's General Catalogue of Artificial Space Objects

Payload data from the LL2 API is new and only contains data for launches in 2026. Hence, I relied on Jonathan McDowell's GCAT for payload data. Data from this source was available, and hence downloaded, as a `.tsv` file.

In [107]:
url_gcat='https://planet4589.org/space/gcat/tsv/launch/Falcon9.tsv'
file_name_gcat = 'mcdowell-gcat-launch-data.tsv'
download_launch_data_static(url_gcat, file_name_gcat)

mcdowell-gcat-launch-data.tsv already exists


## Merging the data

After collecting the raw data, I have merged the three sources into one `csv` file and saved it in `data/interim` for later processing.